<a href="https://colab.research.google.com/github/Diego-1099/Colabfiles/blob/main/Tarea_4_Redes_Sem%C3%A1nticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---



# **Práctica 4. Redes Semánticas**

**Paso 1. Diseño de la Red Semántica**

Para representar el conocimiento del Agente Virtual, se diseño una red semántica basada en las relaciones estándar de ConceptNet. Se definieron más de 20 nodso únicos que abarcan la infraestructura técnica (ERP, WhatsApp), los objetos de negocio (Clientes, Productos) y la lógica comercial. Se garantizó un mínimo de 60 aristas en la implementación.

**2. Jerarquía principal (es_un) -3 niveles mínimos sin ciclos**

Se implementaron varias jerarquías de herencia estricta. Ejemplos:


*   **Jumex_Lata** es_un **Bebida** es_un **Producto**
*   **Grupo_Mexicano** es_un **Cliente_B2B** es_un **Entidad_Comercial**
*   **Memoria_de_Elefante** es_un **Base_de_Datos** es_un **Componente_Sistema**


**3. Relaciones de ConceptNet utilizadas**

Además de la herencia, la red integra las siguientes relaciones semánticas:

*   **parte_de (PartOf):** Indica composición estructural. Ejemplo: **Filtro_de_Belleza** parte_de **Agente_Virtual**.
*   **usado_para (UsedFor):** Indica función o propósito. Ejemplo: **WhatsApp_API** usado_para **Interaccion_Cliente**.
*   **tiene_propiedad (HasProperty):** Atributos descriptivos. Ejemplo: **Cliente_B2B** tiene_propiedad **Compra_Mayoreo**.
*   **relacionado_con (RelatedTo):** Asociaciones generales. Ejemplo: **Venta** relacionado_con **Inventario**.






---



# **Paso 2. Implementación en Python**

In [2]:
class RedSemantica:
  def __init__(self):
    self.grafo = {}

  def agregar_relacion(self, origen, relacion, destino):
    if origen not in self.grafo:
      self.grafo[origen] = []
    if destino not in self.grafo:
      self.grafo[destino] = []
    self.grafo[origen].append((relacion, destino))

  # --- INFERENCIA POR HERENCIA ---

  def obtener_ancestros(self, nodo, visitados = None):
    if visitados is None:
      visitados = set()
    ancestros = []

    if nodo in self.grafo:
      for relacion, destino in self.grafo[nodo]:
          if relacion == "es_un" and destino not in visitados:
            visitados.add(destino)
            ancestros.append(destino)
            ancestros.extend(self.obtener_ancestros(destino, visitados))
    return list(set(ancestros))

  def obtener_propiedades_heredadas(self, nodo):
    propiedades = []
    nodos_a_revisar = [nodo] + self.obtener_ancestros(nodo)

    for n in nodos_a_revisar:
      if n in self.grafo:
        for relacion, destino in self.grafo[n]:
          if relacion == "tiene_propiedad":
            propiedades.append(f"{destino} (heredado de {n})" if n != nodo else destino)
    return propiedades

  def obtener_usados_para_heredados(self, nodo):
    usados_para = []
    nodos_a_revisar = [nodo] + self.obtener_ancestros(nodo)

    for n in nodos_a_revisar:
      if n in self.grafo:
        for relacion, destino in self.grafo[n]:
          if relacion == "usado_para":
            usados_para.append(f"{destino} (heredado de {n})" if n != nodo else destino)
    return usados_para

# ==============================================================================
# CONSTRUCCIÓN DE LA RED (Dominio: Agente Virtual)
# ==============================================================================

rs = RedSemantica()

# Lista de 60 aristas cubriendo las 5 relaciones de ConceptNet
relaciones = [
    # --- Jerarquía es_un (Mínimo 3 niveles) ---
    ('Jumex_Lata', 'es_un', 'Bebida_Frutal'),
    ('Bebida_Frutal', 'es_un', 'Bebida'),
    ('Coca_Cola', 'es_un', 'Refresco'),
    ('Refresco', 'es_un', 'Bebida'),
    ('Agua_Ciel', 'es_un', 'Agua_Embotellada'),
    ('Agua_Embotellada', 'es_un', 'Bebida'),
    ('Bebida', 'es_un', 'Producto_Inventario'),
    ('Producto_Inventario', 'es_un', 'Articulo_Comercial'),
    ('Grupo_Mexicano', 'es_un', 'Cliente_B2B'),
    ('Tienda_Abarrotes', 'es_un', 'Cliente_B2B'),
    ('Cliente_B2B', 'es_un', 'Entidad_Comercial'),
    ('Entidad_Comercial', 'es_un', 'Organizacion'),
    ('Memoria_de_Elefante', 'es_un', 'Base_de_Datos'),
    ('Base_de_Datos', 'es_un', 'Componente_Sistema'),
    ('WhatsApp_API', 'es_un', 'Interfaz_Comunicacion'),
    ('Interfaz_Comunicacion', 'es_un', 'Componente_Sistema'),
    ('Filtro_de_Belleza', 'es_un', 'Componente_Sistema'),
    ('Agente_Virtual', 'es_un', 'Software'),
    ('ERP', 'es_un', 'Software'),
    ('Dashboard_KPI', 'es_un', 'Componente_Sistema'),
    ('Backend_Python', 'es_un', 'Infraestructura_Servidor'),

    # --- parte_de ---
    ('Filtro_de_Belleza', 'parte_de', 'Agente_Virtual'),
    ('Memoria_de_Elefante', 'parte_de', 'Agente_Virtual'),
    ('WhatsApp_API', 'parte_de', 'Agente_Virtual'),
    ('Backend_Python', 'parte_de', 'Agente_Virtual'),
    ('Dashboard_KPI', 'parte_de', 'Agente_Virtual'),
    ('Agente_Virtual', 'parte_de', 'Ecosistema_Zaro'),
    ('ERP', 'parte_de', 'Ecosistema_Zaro'),
    ('Venta', 'parte_de', 'Interaccion_Cliente'),
    ('Catalogo', 'parte_de', 'ERP'),
    ('Inventario', 'parte_de', 'ERP'),

    # --- usado_para ---
    ('WhatsApp_API', 'usado_para', 'Mandar_Mensajes'),
    ('Agente_Virtual', 'usado_para', 'Automatizar_Ventas'),
    ('Memoria_de_Elefante', 'usado_para', 'Persistencia_de_Sesion'),
    ('Memoria_de_Elefante', 'usado_para', 'Guardar_Carritos'),
    ('Filtro_de_Belleza', 'usado_para', 'Traducir_Descripciones_Tecnicas'),
    ('Dashboard_KPI', 'usado_para', 'Analisis_Comportamiento'),
    ('ERP', 'usado_para', 'Gestionar_Inventario'),
    ('Backend_Python', 'usado_para', 'Procesar_Logica'),
    ('Bebida', 'usado_para', 'Hidratar'),
    ('Jumex_Lata', 'usado_para', 'Disfrutar_Sabor'),
    ('Producto_Inventario', 'usado_para', 'Vender'),
    ('Cliente_B2B', 'usado_para', 'Comprar_Mayoreo'),

    # --- tiene_propiedad ---
    ('Producto_Inventario', 'tiene_propiedad', 'Tiene_Precio'),
    ('Producto_Inventario', 'tiene_propiedad', 'Tiene_Stock'),
    ('Bebida', 'tiene_propiedad', 'Liquido'),
    ('Jumex_Lata', 'tiene_propiedad', 'Sabor_Fruta'),
    ('Coca_Cola', 'tiene_propiedad', 'Gaseoso'),
    ('Agua_Ciel', 'tiene_propiedad', 'Natural'),
    ('Cliente_B2B', 'tiene_propiedad', 'Compra_Volumen'),
    ('Entidad_Comercial', 'tiene_propiedad', 'Tiene_RFC'),
    ('Grupo_Mexicano', 'tiene_propiedad', 'Cliente_Frecuente'),
    ('Tienda_Abarrotes', 'tiene_propiedad', 'Cliente_Moroso'),
    ('Agente_Virtual', 'tiene_propiedad', 'Disponible_24_7'),
    ('Memoria_de_Elefante', 'tiene_propiedad', 'Persistente'),

    # --- relacionado_con ---
    ('Agente_Virtual', 'relacionado_con', 'ERP'),
    ('Venta', 'relacionado_con', 'Inventario'),
    ('Stock_Cero', 'relacionado_con', 'Venta_Perdida'),
    ('Dashboard_KPI', 'relacionado_con', 'Ventas'),
    ('WhatsApp_API', 'relacionado_con', 'Meta_Facebook'),
    ('Filtro_de_Belleza', 'relacionado_con', 'UX_UI'),
    ('Cliente_B2B', 'relacionado_con', 'Descuento'),
    ('Grupo_Mexicano', 'relacionado_con', 'Bebida_Frutal'),
    ('Tienda_Abarrotes', 'relacionado_con', 'Agua_Embotellada'),
    ('ERP', 'relacionado_con', 'Backend_Python')
]

# Insertar relaciones al grafo

for origen, relacion, destino in relaciones:
  rs.agregar_relacion(origen, relacion, destino)

# ==============================================================================
# PRUEBAS DE INFERENCIA
# ==============================================================================

nodo_prueba = "Jumex_Lata"
print(f"--- Consultas para el nodo '{nodo_prueba}' ---")
print(f"Ancestros: {rs.obtener_ancestros(nodo_prueba)}")
print(f"Propiedades heredadas: {rs.obtener_propiedades_heredadas(nodo_prueba)}")
print(f"Usado para (heredado): {rs.obtener_usados_para_heredados(nodo_prueba)}")




--- Consultas para el nodo 'Jumex_Lata' ---
Ancestros: ['Bebida_Frutal', 'Producto_Inventario', 'Bebida', 'Articulo_Comercial']
Propiedades heredadas: ['Sabor_Fruta', 'Tiene_Precio (heredado de Producto_Inventario)', 'Tiene_Stock (heredado de Producto_Inventario)', 'Liquido (heredado de Bebida)']
Usado para (heredado): ['Disfrutar_Sabor', 'Vender (heredado de Producto_Inventario)', 'Hidratar (heredado de Bebida)']




---



# **Paso 3. Reflexión y Análisis de la Red Semántica**

1. **Ambigüedad:** Identifiqué términos como "Cliente_Moroso", "Producto_Premium" y "Venta_Perdida". Decidí sus significado anclándolos a atributos medibles del ERP: "Moroso" significa tener facturas vencidad > 30 días, y "Premium" se refiere a un margen de ganancia alto, eliminando la ambigüedad semántica del lenguaje humano.

2. **Granularidad:** Elegí un nivel intermedio, enfocado en entidades comerciales. Si la red fuera demasiado fina, se volvería computacionalmente inmanejable e irrelevante para ventas. Si fuera muy restringida, el bot no podría distinguir entre vender un producto u otro.

3. **Calidad de relaciones:** Relaciones de ConceptNet como **DesireOf** (Deseo de) o **CapableOf** (Capaz de) resultaron poco útiles. Por ejemplo, decir que un **Cliente_B2B** CapableOf **Comprar** es demasiado abstracto y no aporta valor lógico para un sistema transaccional donde lo que importa son los inventarios y las deudas.

4. **Contexto y temporalidad:** El atributo **Tiene_Stock** y **Cliente_Moroso** son altamente temporales (cambian a diario). Una red semántica clásica es estática, por lo que representarlo rígidamente aquí es una limitación. Se requeriría actualizar los nodos en tiempo real desde la "Memoria de Elefante" o usar redes dinámicas temporales.

5. **Inferencia limitada:** Solo con la herencia **es_un** no puedo inferir conclusiones numéricas (cuánto es el total de un carrito) ni ejecutar validaciones transaccionales (como bloquear una venta). Me faltaría el motor de Lógica de Primer Orden (reglas del sistema anterior) para operar la lógica de negocio.

6. **Conflictos:** Si una empresa hereda la propiedad **Tiene_descuento** por ser Mayorista, pero también hereda **Venta_Bloqueada** por ser Morosa, el grafo entra en conflicto. Lo resolvería implementando un sistema de pesos y prioridades (ontología formal) donde las reglas de riesgo financiero tengan precedencia sobre las comerciales.

7. **Explicabilidad:** El grafo hace que la IA no sea una "caja negra". Si un cliente pide un producto agotado y el bot recomienda otro, el grafo explica la decisión porque ambos nodos comparten un ancestro común (ej. **Agua_Embotellada) y la propiedad **Hidratar**, justificando semánticamente la recomendación.

8. **Escalabilidad:** Si la red crece a 10,000 nodos, la búsqueda en diccionarios de Python sería insostenible. Tendría que migrar la estructura a una base de datos orientada a grafos reales, e implementar índices de búsqueda vectoriales o lenguajes de consulta como Cypher.

9. **Validez:** Para validar que la red es correcta, necesitaría contrastarla con los expertos del dominio y asegurar que la jerarquía coincida al 100% con el esquema de la base de datos relacional de productos (la taxonomía oficial del ERP).



---



# **Paso 4. Búsqueda en Google Académico**

Referencia:

*   Enhancing Generative AI Chatbot Accuracy Using Knowledge Graph (2024). Investigación sobre la integración de bases de datos orientadas o grafos con Modelos Fundacionales.

**Resumen del Artículo:**
El problema que aborda el estudio es la tendencia de los chatbots basados en Inteligencia Artificial Generativa a sufrir de inexactitudes y "alucinaciones" al responder preguntas cerradas o basadas en datos estructurados. El objetivo fue diseñar un marco robusto que entregue respuestas precisas y apropiadas al contexto. La solución propuesta fue integrar Grafos de Conocimiento (Knowledge Graphs usando Neo4j) directamente con grandes modelos de lenguaje (LLMs) mediante una arquitectura RAG (Retrieval-Augmented Generation). Los autores concluyen que estructurar la información del negocio en nodos y relaciones (similar a una red semántica) mejora dramáticamente la precisión y la comprensión contextual del agente, permitiéndole interactuar de forma amigable sin inventar datos.

**Similitudes y aplicación al Proyecto de Tesis:**
Este artículo valida la evolución futura del Agente Virtual de Ventas. Actualmente, mi sistema usa "Filtros de Belleza" y una "Memoria de Elefante" estructurada. Si el Agente Virtual integra un LLM en WhatsApp para entender al usuario de forma natural, correría el riesgo de alucinar precios o inventarios. Aplicar la solución del artículo (convertir esta Red Semántica de catálogo y clientes en un Knowledge Graph de Neo4j) me permitiría crear un sistema neurosimbólico: el LLM conversaría fluidamente, pero consultaría el grafo de forma determinista para asegurar que el **stock** y el **precio** son 100% reales antes de enviar el mensaje al cliente.



---

